# move_detection: stable frames + move inference (M5, optional)

Move detection from a stream of warped top-down board images. Pipeline per move:
1. `StableFrameDetector` watches frame-to-frame differences and fires once the board is
   still again after a period of motion (the player's hand came in and left).
2. The stable frame is cut into 64 crops and classified -> (64, 13) probabilities.
3. `infer_move()` scores every legal move of the previous position (plus "no move")
   against those probabilities and returns the most likely one. Because candidates come
   from python-chess, castling, en passant and promotion need no special cases.

Status: `infer_move` / `changed_squares` are implemented; `StableFrameDetector` has simple
working logic but untuned thresholds; `MoveTracker` and `evaluate_on_game` are skeletons.
Evaluated on recorded games by `run_move_detection.ipynb`.

Milestone: M5 (optional).

**TODO** (see `docs/M5_TODO.md`)
- [ ] record games (video/frames + PGN)
- [ ] tune `StableFrameDetector` / `frame_difference`
- [ ] build the `board_proba` callable (homography warp -> crops -> M3 model)
- [ ] `MoveTracker.process_frame`
- [ ] `evaluate_on_game`

In [ ]:
from __future__ import annotations

from collections.abc import Callable, Iterable, Sequence
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path

import chess
import numpy as np

%run ./labels.ipynb
# board_assembly.ipynb (M4) is only needed for the optional cross-check in MoveTracker;
# %run it in the run notebook if you use it.

## Stable frames
State machine IDLE -> MOTION -> SETTLING -> STABLE -> IDLE on the mean absolute frame
difference.

In [ ]:
class State(str, Enum):
    IDLE = "idle"          # board static, nothing happening
    MOTION = "motion"      # large frame differences (hand over the board)
    SETTLING = "settling"  # differences small again, waiting for settle_frames in a row
    STABLE = "stable"      # emitted for exactly one frame after settling; then back to IDLE


def frame_difference(a: np.ndarray, b: np.ndarray) -> float:
    """Mean absolute difference between two frames, in [0, 255] intensity units.

    Color frames are averaged to grayscale first. TODO(M5): consider downsampling or a
    Gaussian blur first to suppress sensor noise / flicker; try a per-square max instead
    of a global mean so a small hand at the board edge is not averaged away.
    """
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    if a.shape != b.shape:
        raise ValueError(f"frame shapes differ: {a.shape} vs {b.shape}")
    if a.ndim == 3:
        a, b = a.mean(axis=2), b.mean(axis=2)
    return float(np.abs(a - b).mean())


class StableFrameDetector:
    """State machine IDLE -> MOTION -> SETTLING -> STABLE -> IDLE.

    Feed frames with update(frame); it returns True exactly once per motion episode, on the
    frame where the board has been still for `settle_frames` consecutive frames after at
    least `min_motion_frames` frames of motion. `stable_frame` / `reference_frame` hold the
    frames after and before the episode.

    Defaults are guesses. TODO(M5): tune diff_threshold / settle_frames on recorded
    video (plot frame_difference over time for a few games), and decide whether to add
    an explicit hand detector (e.g. skin-color mask or "occupancy outside the start/end
    squares") so slow hand movements are not mistaken for a still board.
    """

    def __init__(self, diff_threshold: float = 4.0, settle_frames: int = 10,
                 min_motion_frames: int = 2) -> None:
        self.diff_threshold = diff_threshold
        self.settle_frames = settle_frames
        self.min_motion_frames = min_motion_frames
        self.reset()

    def reset(self) -> None:
        self.state = State.IDLE
        self.prev_frame: np.ndarray | None = None
        self.reference_frame: np.ndarray | None = None  # last still frame before motion
        self.stable_frame: np.ndarray | None = None     # frame at which STABLE fired
        self.last_diff = 0.0
        self._motion_count = 0
        self._still_count = 0

    def update(self, frame: np.ndarray) -> bool:
        if self.prev_frame is None:
            self.prev_frame = self.reference_frame = frame
            return False
        diff = frame_difference(self.prev_frame, frame)
        self.last_diff = diff
        moving = diff > self.diff_threshold
        fired = False

        if self.state in (State.IDLE, State.STABLE):
            if moving:
                self.state, self._motion_count = State.MOTION, 1
            else:
                self.state = State.IDLE
                self.reference_frame = frame
        elif self.state == State.MOTION:
            if moving:
                self._motion_count += 1
            elif self._motion_count >= self.min_motion_frames:
                self.state, self._still_count = State.SETTLING, 1
            else:  # a one-frame blip (noise, flicker): ignore it
                self.state = State.IDLE
        elif self.state == State.SETTLING:
            if moving:
                self.state = State.MOTION
                self._motion_count += 1
            else:
                self._still_count += 1

        if self.state == State.SETTLING and self._still_count >= self.settle_frames:
            self.state = State.STABLE
            self.stable_frame = frame
            fired = True

        self.prev_frame = frame
        return fired

## Move inference
Score = sum over squares of log P(square has the label the candidate position implies).
`ambiguity_margin` is in nats: 2.0 means the best candidate is ~7x more likely than the
runner-up.

In [ ]:
@dataclass
class MoveResult:
    move: chess.Move | None        # None = board judged unchanged
    san: str | None                # SAN in the previous position, None for "no move"
    score: float                   # log-likelihood of the best candidate
    runner_up: chess.Move | None   # second-best candidate (None may mean "no move")
    margin: float                  # score(best) - score(runner_up), >= 0
    ambiguous: bool                # margin below threshold -> ask for confirmation / wait
    ranking: list[tuple[chess.Move | None, float]] = field(default_factory=list, repr=False)


def placement_log_likelihood(labels: Sequence[str], log_probs: np.ndarray) -> float:
    """Sum over squares of log P(square has this label)."""
    idx = np.fromiter((CLASS_TO_INDEX[x] for x in labels), dtype=np.int64, count=64)
    return float(log_probs[np.arange(64), idx].sum())


def infer_move(prev_board: chess.Board, square_probs_after: np.ndarray, *,
               include_null: bool = True, ambiguity_margin: float = 2.0,
               eps: float = 1e-6) -> MoveResult:
    """Most likely legal move from prev_board given (64, 13) probabilities of the new frame.

    Each candidate (every legal move, plus "no move" if include_null) is pushed onto a copy
    of prev_board and scored by the log-likelihood of the resulting piece placement, treating
    squares as independent. `ambiguity_margin` is in nats: 2.0 means the best candidate is
    ~7x more likely than the runner-up.

    TODO(M5): if the side to move is uncertain (e.g. tracking started mid-game), also try
    the null move to flip turns; consider two-ply candidates when a stable frame was missed.
    """
    probs = np.asarray(square_probs_after, dtype=np.float64)
    if probs.shape != (64, NUM_CLASSES):
        raise ValueError(f"expected (64, {NUM_CLASSES}) probabilities, got {probs.shape}")
    log_probs = np.log(np.clip(probs, eps, 1.0))

    ranking: list[tuple[chess.Move | None, float]] = []
    if include_null:
        ranking.append((None, placement_log_likelihood(board_to_labels(prev_board), log_probs)))
    board = prev_board.copy(stack=False)
    for move in prev_board.legal_moves:
        board.push(move)
        ranking.append((move, placement_log_likelihood(board_to_labels(board), log_probs)))
        board.pop()
    if not ranking:  # game over and include_null=False
        return MoveResult(None, None, float("-inf"), None, 0.0, True)

    ranking.sort(key=lambda t: t[1], reverse=True)
    best, best_score = ranking[0]
    runner_up, second = ranking[1] if len(ranking) > 1 else (None, float("-inf"))
    margin = best_score - second
    return MoveResult(
        move=best,
        san=prev_board.san(best) if best is not None else None,
        score=best_score,
        runner_up=runner_up,
        margin=margin,
        ambiguous=margin < ambiguity_margin,
        ranking=ranking,
    )


def changed_squares(prev_labels: Sequence[str], new_labels: Sequence[str]) -> list[str]:
    """Names of squares whose label differs, in square-index order (a1 first)."""
    if len(prev_labels) != 64 or len(new_labels) != 64:
        raise ValueError("expected 64 labels each")
    return [SQUARE_NAMES[i] for i, (a, b) in enumerate(zip(prev_labels, new_labels)) if a != b]

## Tracking a game

In [ ]:
ProbaFn = Callable[[np.ndarray], np.ndarray]  # warped frame -> (64, 13) probabilities


@dataclass
class MoveEvent:
    frame_index: int
    result: MoveResult
    fen_after: str


class MoveTracker:
    """Detector + classifier + infer_move over a frame stream, maintaining the game board.

    `board_proba` maps a warped top-down frame to (64, 13) probabilities; build it from
    capture.ipynb's extract_all_crops + an M3 model's predict_proba.
    """

    def __init__(self, board_proba: ProbaFn, start_board: chess.Board | None = None,
                 detector: StableFrameDetector | None = None,
                 ambiguity_margin: float = 2.0) -> None:
        self.board_proba = board_proba
        self.board = start_board.copy() if start_board is not None else chess.Board()
        self.detector = detector or StableFrameDetector()
        self.ambiguity_margin = ambiguity_margin
        self.events: list[MoveEvent] = []
        self._frame_index = 0

    def process_frame(self, frame: np.ndarray) -> MoveEvent | None:
        """Feed one warped frame; return a MoveEvent when a move was recognized."""
        self._frame_index += 1
        if not self.detector.update(frame):
            return None
        # TODO(M5):
        #   probs = self.board_proba(frame)
        #   result = infer_move(self.board, probs, ambiguity_margin=self.ambiguity_margin)
        #   - result.move is None -> hand passed over / piece adjusted; no event
        #   - result.ambiguous -> keep the frame, wait for the next stable frame or flag
        #     for the user/robot instead of pushing a guess
        #   - else push onto self.board, append MoveEvent(self._frame_index, result,
        #     self.board.fen()) to self.events and return it
        #   Optionally cross-check with assemble(probs) from board_assembly.ipynb.
        raise NotImplementedError("MoveTracker.process_frame: see TODO(M5)")

    def run(self, frames: Iterable[np.ndarray]) -> list[MoveEvent]:
        """Process a whole stream (e.g. frames read from a video and warped by homography)."""
        for frame in frames:
            self.process_frame(frame)
        return self.events


def evaluate_on_game(pgn_path: str | Path, frames_dir: str | Path,
                     board_proba: ProbaFn | None = None, **tracker_kwargs) -> dict:
    """Replay one recorded game through MoveTracker and score it against its PGN.

    Inputs: the game's PGN (ground truth) and a directory of warped frames in time order
    (e.g. data/crops/<session>/frames/ or frames extracted from the game video).

    Returns a metrics dict (run_move_detection.ipynb writes it to
    results/<run_name>/move_metrics.json):
        n_moves             plies in the PGN
        move_accuracy       fraction of PGN plies recognized correctly, aligned in order
                            (a missed or wrong move counts as an error; after an error the
                            tracker is resynced to the true position so one mistake does
                            not cascade - report the unsynced "game completed" flag too)
        missed              PGN plies with no detected move
        spurious            detected moves with no corresponding ply
        ambiguous           detections flagged ambiguous
        special_moves       accuracy restricted to castling / en passant / promotion plies
        game_completed      True if the tracker reproduced the full game without resync

    TODO(M5): load PGN with chess.pgn.read_game, load frames (sorted by name), build the
    tracker, align detected moves with mainline moves, compute the numbers above.
    """
    raise NotImplementedError("evaluate_on_game: see docstring TODO(M5)")

## Self-check
(was tests/test_move_detection.py)

TODO notes from skipped tests (need a recorded game, PGN + frames):
- `evaluate_on_game` with a `board_proba` that returns one-hot probabilities of the true
  position gives move_accuracy == 1.0 (render/collect frames for a short PGN).
- `evaluate_on_game` reports special-move accuracy (castling, en passant, promotion).

In [ ]:
def _md_one_hot(board: chess.Board, noise: float = 0.0) -> np.ndarray:
    probs = np.full((64, NUM_CLASSES), noise / NUM_CLASSES)
    for sq, lab in enumerate(board_to_labels(board)):
        probs[sq, CLASS_TO_INDEX[lab]] += 1.0 - noise
    return probs


def _md_after(fen: str, uci: str):
    board = chess.Board(fen)
    nxt = board.copy()
    nxt.push_uci(uci)
    return board, _md_one_hot(nxt)


# infer_move, including castling, en passant, promotion
for _fen, _uci, _san in [
    (chess.STARTING_FEN, "e2e4", "e4"),
    ("r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1", "e1g1", "O-O"),
    ("rnbqkbnr/1pp1pppp/p7/3pP3/8/8/PPPP1PPP/RNBQKBNR w KQkq d6 0 3", "e5d6", "exd6"),
    ("8/P7/8/8/8/8/k6K/8 w - - 0 1", "a7a8q", "a8=Q+"),
]:
    _board, _probs = _md_after(_fen, _uci)
    _res = infer_move(_board, _probs)
    assert _res.move == chess.Move.from_uci(_uci) and _res.san == _san, (_uci, _res)
    assert not _res.ambiguous and _res.margin > 0

# queenside castling distinct from kingside
_board, _probs = _md_after("r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1", "e1c1")
assert infer_move(_board, _probs).san == "O-O-O"

# unchanged board is "no move"
_res = infer_move(chess.Board(), _md_one_hot(chess.Board()))
assert _res.move is None and _res.san is None and not _res.ambiguous

# noisy probabilities still pick the move
_board = chess.Board()
_nxt = _board.copy()
_nxt.push_uci("g1f3")
assert infer_move(_board, _md_one_hot(_nxt, noise=0.5)).move == chess.Move.from_uci("g1f3")

# bad shape raises
try:
    infer_move(chess.Board(), np.zeros((64, 12)))
    raise AssertionError("expected ValueError")
except ValueError:
    pass

# changed squares
_board = chess.Board()
_prev = board_to_labels(_board)
_board.push_uci("e2e4")
assert changed_squares(_prev, board_to_labels(_board)) == ["e2", "e4"]

# stable frame detector on synthetic frames
_rng = np.random.default_rng(0)
_still_a = np.full((64, 64), 100.0)
_still_b = _still_a.copy()
_still_b[10:20, 10:20] = 200  # a piece moved
_frames = (
    [_still_a + _rng.normal(0, 0.5, _still_a.shape) for _ in range(5)]       # static
    + [_rng.uniform(0, 255, _still_a.shape) for _ in range(4)]               # hand
    + [_still_b + _rng.normal(0, 0.5, _still_a.shape) for _ in range(12)]    # static
)
_det = StableFrameDetector(diff_threshold=4.0, settle_frames=5)
_fired, _refs = [], []
for _i, _f in enumerate(_frames):
    if _det.update(_f):
        _fired.append(_i)
        _refs.append(_det.reference_frame)
# first still frame after motion is index 9 -> 1 diff vs hand frame (moving),
# then still diffs from index 10; 5 in a row -> fires at index 14
assert _fired == [14]
assert _det.stable_frame is _frames[14]
assert _refs[0] is _frames[4]  # last still frame before the hand appeared
assert _det.state == State.IDLE
print("move_detection OK")